# Northstar Triage Copilot — app

Loads the six trained model bundles from `models/` and serves them. **Trains nothing.** Two tabs: a triage screen for the agent, and an exploration screen for inspecting how each model behaves (metrics, confusion, held-out predictions).

## 1. Load bundles

In [ ]:
import os, sys, warnings, html
warnings.filterwarnings("ignore")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import numpy as np, pandas as pd, joblib
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import northstar_prep as P

KEYS  = ["m1_case_type","m2_category","m3_priority_direct","m3_priority_review","m4_escalation","m5_route"]
ORDER = ["m1_case_type","m2_category","m3_priority_direct","m3_priority_review","m5_route","m4_escalation"]
B = {}
for k in KEYS:
    path = os.path.join(P.MODELS_DIR, f"{k}.joblib")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing — run training/train_{k}.ipynb first")
    B[k] = joblib.load(path)
for k in ORDER:
    print(f'{k:20s} {B[k]["label"]:22s} {B[k]["meta"]["approach"]}')

## 2. Prediction cascade

In [ ]:
PRIORITIES = B["m3_priority_review"]["meta"]["priorities"]
RANK = {p:i for i,p in enumerate(PRIORITIES)}

def _row(base, extra=None): return pd.DataFrame([{**base, **(extra or {})}])
def _order(key, row):
    b=B[key]; m=b["model"]; p=m.predict_proba(row[b["features"]])[0]; o=p.argsort()[::-1]
    return [(str(m.classes_[i]), float(p[i])) for i in o]

def predict_case(summary, channel, plan, month, tenure, tags, agent_priority=None):
    tags=(tags or "").strip()
    base={"case_summary":summary or "", "tags":tags, "channel":channel, "plan_tier":plan,
          "month_num":int(month), "customer_tenure_months":int(tenure),
          "n_tags":0 if not tags else len(tags.split(";"))}
    ct  = _order("m1_case_type", _row(base))[0]
    cat = _order("m2_category",  _row(base))[0]
    pri_d = _order("m3_priority_direct", _row(base))[0]
    pri_r = _order("m3_priority_review", _row(base, {"case_type":ct[0]}))[0]
    rt  = _order("m5_route", _row(base))[0]
    b4=B["m4_escalation"]; m4=b4["model"]
    p4=m4.predict_proba(_row(base, {"category":cat[0]})[b4["features"]])[0]
    esc_p=float(p4[list(m4.classes_).index(True)]); esc_flag=esc_p>=b4["threshold"]
    out={"case_type":ct, "category":cat, "route":rt, "priority_direct":pri_d, "priority_review":pri_r,
         "pri_agree":pri_d[0]==pri_r[0], "esc_prob":esc_p, "esc_flag":esc_flag,
         "review":rt[1] < B["m5_route"].get("review_threshold",0.0),
         "terms":_top_terms("m1_case_type", _row(base), 3)}
    if agent_priority in RANK:
        out["agent_priority"]=agent_priority
        out["priority_flag"]=abs(RANK[agent_priority]-RANK[pri_r[0]]) >= B["m3_priority_review"]["meta"]["flag_gap"]
    return out

_STOP=set("the a an and or of to in for with on at is are be was were this that it its my our your i we "
          "you they he she as by from not no cannot can keeps after getting get got need want please thanks "
          "cheers hi hello there their them has have had do does did been being but so if then when which who "
          "what why how able keep still just also very more most".split())
def _top_terms(key,row,k=3):
    b=B[key]; m=b["model"]; prep,clf=m.named_steps["prep"],m.named_steps["clf"]
    names=prep.get_feature_names_out(); x=prep.transform(row[b["features"]]); xr=np.asarray(x.todense()).ravel()
    ci=list(clf.classes_).index(m.predict(row[b["features"]])[0])
    coef=clf.coef_; c=coef[ci] if coef.shape[0]>1 else (coef[0] if ci==1 else -coef[0])
    contrib=c*xr; pref="case_summary_w__"
    idx=[i for i in range(len(names)) if names[i].startswith(pref)]; idx.sort(key=lambda i:contrib[i],reverse=True)
    out=[]
    for i in idx:
        if contrib[i]<=0: break
        term=" ".join(w for w in names[i].replace(pref,"").split() if w not in _STOP and len(w)>2)
        if term and term not in out: out.append(term)
        if len(out)>=k: break
    return out
print(predict_case("Cannot connect the Xero integration, OAuth token fails with a 500 error","in_app","Enterprise",6,30,"xero;integration"))

## 3. Result card

In [ ]:
THR=B["m4_escalation"]["threshold"]
def band(p):
    if p>=THR:   return "High","#dc2626","#fef2f2"
    if p>=THR/2: return "Medium","#d97706","#fffbeb"
    return "Low","#059669","#ecfdf5"
def explain(o):
    bits=[]
    if o["terms"]: bits.append(", ".join(o["terms"]) + " in summary")
    bits.append(f'{o["case_type"][0].replace("_"," ")} language')
    bits.append("flagged above the escalation threshold" if o["esc_flag"]
                else ("elevated but below the escalation threshold" if o["esc_prob"]>=THR/2
                      else "below the escalation threshold"))
    return " + ".join(bits)
def render_card(o):
    lv,col,bg=band(o["esc_prob"])
    agree='' if o["pri_agree"] else ' <span class="tag warn">models differ</span>'
    rows=[("Case type", f'{o["case_type"][0].replace("_"," ").title()} <span class="muted">{o["case_type"][1]:.0%}</span>'),
          ("Category",  f'{o["category"][0].replace("_"," ").title()} <span class="muted">{o["category"][1]:.0%}</span>'),
          ("Priority · direct", f'{o["priority_direct"][0]} <span class="muted">{o["priority_direct"][1]:.0%}</span>'),
          ("Priority · review", f'{o["priority_review"][0]} <span class="muted">{o["priority_review"][1]:.0%}</span>{agree}'),
          ("Suggested route", o["route"][0].title() + (' <span class="tag warn">needs human review</span>' if o["review"] else ''))]
    tr="".join(f'<tr><td class="k">{k}</td><td class="v">{v}</td></tr>' for k,v in rows)
    risk=f'<tr><td class="k">Risk</td><td class="v"><span class="pill" style="color:{col};background:{bg}">{lv} escalation risk · {o["esc_prob"]:.0%}</span></td></tr>'
    extra=""
    if o.get("priority_flag"):
        extra=f'<tr><td class="k">Priority check</td><td class="v"><span class="pill" style="color:#dc2626;background:#fef2f2">review: you set {o["agent_priority"]}, estimate is {o["priority_review"][0]}</span></td></tr>'
    why=f'<tr><td class="k">Why</td><td class="v why">{html.escape(explain(o))}</td></tr>'
    return f'<div class="rescard"><table class="res">{tr}{risk}{extra}{why}</table></div>'
print(render_card(predict_case("Dashboard is down for all users, 500 errors","phone","Pro",9,36,"outage"))[:70],"...")

## 4. Exploration helpers

In [ ]:
def model_cards():
    g=""
    for k in ORDER:
        b=B[k]; meta=b["meta"]; m=b["metrics"]
        core=["accuracy","macro_F1","weighted_F1","cohen_kappa","ROC_AUC","PR_AUC"]
        met=" · ".join(f'{c} <b>{m[c]}</b>' for c in core if c in m)
        extra_keys=[kk for kk in m if kk not in core+["n_test","baseline_acc"]]
        extra=" · ".join(f'{kk} <b>{m[kk]}</b>' for kk in extra_keys)
        g+=(f'<div class="mcard"><div class="mh">{k.split("_")[0].upper()} — {b["label"]}'
            f'<span class="chip">{meta["contributor"]}</span></div><div class="mb">'
            f'<div class="spec"><span>Approach</span>{meta["approach"]}</div>'
            f'<div class="spec"><span>Inputs</span>{meta["inputs"]}</div>'
            f'<div class="spec"><span>Split</span>{meta["split"]}</div>'
            f'<div class="spec"><span>Core</span>{met}</div>'
            + (f'<div class="spec"><span>Signature</span>{extra}</div>' if extra else "")
            + (f'<div class="cav">{meta.get("note","")}</div>' if meta.get("note") else "")
            + "</div></div>")
    return f'<div class="mgrid">{g}</div>'

def metrics_table():
    rows=[]
    for k in ORDER:
        b=B[k]; r={"Model":k.split("_")[0].upper(),"Predicts":b["label"]}; r.update(b["metrics"]); rows.append(r)
    cols=["Model","Predicts","n_test","accuracy","macro_F1","weighted_F1","cohen_kappa","ROC_AUC","PR_AUC","baseline_acc"]
    df=pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]]

def case_choices(key):
    t=B[key]["test"]; return [(f'{r.case_id} — {str(r.case_summary)[:55]}', i) for i,r in t.iterrows()]

def inspect(key, idx):
    if key is None or idx is None: return '<div class="rescard"><p class="note">Pick a model and a case.</p></div>'
    b=B[key]; row=b["test"].iloc[int(idx)]
    actual,pred,conf=row[b["target"]],row["predicted"],row.get("confidence",None)
    ok=str(pred)==str(actual)
    mark='<span class="tag ok">✓ match</span>' if ok else '<span class="tag warn">✗ differs</span>'
    extra=f' · prob {row["prob"]:.0%} vs threshold {b.get("threshold",0):.0%}' if b["kind"]=="binary" else ""
    return (f'<div class="rescard"><p class="sum">“{html.escape(str(row.case_summary)[:300])}”</p>'
            f'<table class="insp"><tr><td class="k">Model</td><td>{key.split("_")[0].upper()} — {b["label"]}</td></tr>'
            f'<tr><td class="k">Predicted</td><td>{html.escape(str(pred))}'+(f' ({conf:.0%})' if conf is not None else '')+f'{extra}</td></tr>'
            f'<tr><td class="k">Actual</td><td>{html.escape(str(actual))}</td></tr>'
            f'<tr><td class="k">Result</td><td>{mark}</td></tr></table>'
            f'<p class="note">Held-out case ({b["meta"]["split"]} split) — never seen in training.</p></div>')

def confusion_html(key):
    b=B[key]; cm=b["confusion"]; labs=cm["labels"]; M=cm["matrix"]
    mx=max((max(r) for r in M), default=1) or 1
    head="".join(f'<th>{html.escape(str(l))[:10]}</th>' for l in labs)
    body=""
    for i,l in enumerate(labs):
        tds=""
        for j in range(len(labs)):
            v=M[i][j]; a=v/mx; bgc=f"rgba(79,70,229,{0.08+0.72*a:.2f})" if v else "#fff"
            tds+=f'<td style="background:{bgc}">{v}</td>'
        body+=f'<tr><th>{html.escape(str(l))[:10]}</th>{tds}</tr>'
    return (f'<div class="rescard"><div class="small muted">rows = actual · columns = predicted</div>'
            f'<table class="cm"><tr><th></th>{head}</tr>{body}</table></div>')

def report_html(key):
    b=B[key]; rep=b["report"]
    rows=""
    for cls,d in rep.items():
        if cls in ("accuracy","macro avg","weighted avg") or not isinstance(d,dict): continue
        rows+=f'<tr><td>{html.escape(str(cls))}</td><td>{d["precision"]:.2f}</td><td>{d["recall"]:.2f}</td><td>{d["f1-score"]:.2f}</td><td>{int(d["support"])}</td></tr>'
    return (f'<div class="rescard"><table class="rep"><tr><th>class</th><th>prec</th><th>rec</th><th>F1</th><th>n</th></tr>{rows}</table></div>')
print("exploration helpers ready |", metrics_table().shape)

## 5. App — modern two-tab UI

In [ ]:
import gradio as gr
CHANNELS=["email","webchat","in_app","phone"]; PLANS=["Free","Standard","Pro","Enterprise"]
LABELS={k:f'{k.split("_")[0].upper()} — {B[k]["label"]}' for k in ORDER}
NB_PATH={k:os.path.join(P.ROOT,"training",f"train_{k}.ipynb") for k in KEYS}
EXAMPLE=["Cannot connect the Xero integration — the OAuth token keeps failing with a 500 error after the latest update",
         "in_app","Enterprise",6,30,"xero;integration;error","(none)"]

CSS="""
:root{--ink:#0f172a;--muted:#64748b;--line:#e2e8f0;--accent:#4f46e5;--bg:#f8fafc;}
.gradio-container{background:var(--bg);}
#wrap{max-width:1080px;margin:auto;}
.hero{background:linear-gradient(120deg,#4f46e5,#0ea5e9);color:#fff;border-radius:18px;padding:22px 26px;margin-bottom:14px;}
.hero h1{margin:0;font-size:24px;font-weight:800;letter-spacing:-.3px;}
.hero p{margin:6px 0 0;opacity:.92;font-size:14px;}
.rescard{border:1px solid var(--line);border-radius:16px;padding:20px 24px;background:#fff;box-shadow:0 1px 2px rgba(15,23,42,.04);}
table.res{width:100%;border-collapse:collapse;}
table.res td{padding:12px 6px;border-bottom:1px solid #f1f5f9;font-size:16px;vertical-align:top;}
table.res tr:last-child td{border-bottom:none;}
td.k{color:var(--muted);width:160px;font-weight:600;font-size:13px;text-transform:uppercase;letter-spacing:.03em;}
td.v{color:var(--ink);} td.v.why{color:#334155;font-size:15px;} .muted{color:var(--muted);font-weight:500;font-size:14px;}
.pill{padding:5px 13px;border-radius:999px;font-weight:700;font-size:14px;}
.tag{padding:2px 9px;border-radius:999px;font-size:12px;font-weight:700;}
.tag.warn{color:#b45309;background:#fffbeb;} .tag.ok{color:#047857;background:#ecfdf5;}
.sum{font-style:italic;color:#334155;margin:0 0 14px;} .note{color:#94a3b8;font-size:12px;margin-top:12px;}
.small{font-size:12px;margin-bottom:6px;}
.mgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:14px;}
.mcard{border:1px solid var(--line);border-radius:14px;overflow:hidden;background:#fff;}
.mh{background:#0f172a;color:#fff;padding:11px 14px;font-weight:700;font-size:14px;display:flex;justify-content:space-between;align-items:center;}
.chip{background:rgba(255,255,255,.16);border-radius:999px;padding:1px 9px;font-size:11px;font-weight:700;}
.mb{padding:13px 15px;font-size:13px;color:var(--ink);line-height:1.5;}
.spec{margin-bottom:6px;} .spec>span{display:inline-block;width:84px;color:var(--muted);font-weight:700;vertical-align:top;}
.cav{color:#92400e;font-size:12px;margin-top:8px;padding-top:8px;border-top:1px dashed var(--line);}
table.insp{width:100%;border-collapse:collapse;} table.insp td{padding:8px 6px;border-bottom:1px solid #f1f5f9;font-size:14px;}
table.cm{border-collapse:collapse;font-size:12px;} table.cm th{color:var(--muted);padding:4px 7px;font-weight:700;}
table.cm td{padding:6px 9px;text-align:center;border:1px solid #f1f5f9;min-width:34px;}
table.rep{width:100%;border-collapse:collapse;font-size:13px;} table.rep th{text-align:left;color:var(--muted);border-bottom:2px solid var(--line);padding:5px;}
table.rep td{padding:5px;border-bottom:1px solid #f1f5f9;}
"""

with gr.Blocks(title="Northstar Triage Copilot", css=CSS, theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    with gr.Column(elem_id="wrap"):
        gr.HTML('<div class="hero"><h1>Northstar Triage Copilot</h1>'
                '<p>Six models behind one screen — classify, route, prioritise and flag risk for a new case.</p></div>')
        with gr.Tabs() as tabs:
            # ---------------- Triage ----------------
            with gr.Tab("Triage a case", id="triage"):
                with gr.Row():
                    with gr.Column(scale=5):
                        summary=gr.Textbox(label="Case summary", lines=4, placeholder="Paste the customer's case summary…")
                        with gr.Row():
                            channel=gr.Dropdown(CHANNELS, value="email", label="Channel")
                            plan=gr.Dropdown(PLANS, value="Standard", label="Plan tier (direct-priority model only)")
                        with gr.Row():
                            month=gr.Dropdown(list(range(1,13)), value=6, label="Month")
                            tenure=gr.Number(value=12, label="Tenure (months)")
                        agent_pri=gr.Dropdown(["(none)"]+PRIORITIES, value="(none)", label="Agent's priority (optional — triggers the review check)")
                        tags=gr.Textbox(label="Tags (optional, ';'-separated)", placeholder="leave blank if unknown")
                        with gr.Row():
                            go=gr.Button("Triage case", variant="primary", scale=2)
                            ex=gr.Button("Load example", scale=1)
                    with gr.Column(scale=5):
                        out=gr.HTML('<div class="rescard"><p class="note">Enter a case and press <b>Triage case</b>, or <b>Load example</b>.</p></div>')
                        explore_btn=gr.Button("Explore how the models work →", variant="secondary")
                def triage(s,c,pl,mo,tn,tg,ap): return render_card(predict_case(s,c,pl,mo,tn,tg,None if ap=="(none)" else ap))
                ins=[summary,channel,plan,month,tenure,tags,agent_pri]
                go.click(triage, ins, out)
                ex.click(lambda: EXAMPLE, None, ins).then(triage, ins, out)

            # ---------------- Exploration ----------------
            with gr.Tab("Explore the models", id="explore"):
                gr.Markdown("### Model cards")
                gr.HTML(model_cards())
                gr.Markdown("### Normalised held-out metrics")
                gr.Dataframe(metrics_table(), interactive=False)
                gr.Markdown("### Inspect a held-out case — predicted vs actual")
                with gr.Row():
                    mdl=gr.Dropdown([(LABELS[k],k) for k in ORDER], label="Model")
                    case=gr.Dropdown([], label="Held-out case")
                insp=gr.HTML('<div class="rescard"><p class="note">Pick a model, then a case.</p></div>')
                mdl.change(lambda k: gr.update(choices=case_choices(k), value=None), mdl, case)
                case.change(inspect, [mdl, case], insp)
                gr.Markdown("### Confusion matrix & per-class report")
                mdl2=gr.Dropdown([(LABELS[k],k) for k in ORDER], label="Model", value="m1_case_type")
                with gr.Row():
                    cmv=gr.HTML(confusion_html("m1_case_type"))
                    repv=gr.HTML(report_html("m1_case_type"))
                mdl2.change(lambda k:(confusion_html(k), report_html(k)), mdl2, [cmv, repv])
                gr.Markdown("### Training notebooks")
                with gr.Row():
                    for k in ORDER:
                        gr.DownloadButton(f"⬇ {k}", value=NB_PATH[k], size="sm")

        explore_btn.click(lambda: gr.Tabs(selected="explore"), None, tabs)

print("app built | models:", len(B))

## 6. Launch

In [ ]:
# demo.launch(server_name="127.0.0.1", server_port=7866, inbrowser=False)